## Pré-processamento e transformação dos dados


1. Importa os dados e verifica estrutura das tabelas

In [79]:
import pandas as pd

dados_labled = pd.read_csv('AgeOfMythology_work.csv')

dados_labled.info()

<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unit         83 non-null     str    
 1   Age          83 non-null     str    
 2   Cost         83 non-null     str    
 3   Pop          83 non-null     str    
 4   DmgHack      83 non-null     str    
 5   DmgPierce    83 non-null     str    
 6   DmgCrush     83 non-null     str    
 7   Range        83 non-null     str    
 8   HP           83 non-null     int64  
 9   ArmorHack    83 non-null     str    
 10  ArmorPierce  83 non-null     str    
 11  ArmorCrush   83 non-null     str    
 12  Speed        83 non-null     float64
 13  LOS          83 non-null     int64  
 14  Train time   83 non-null     str    
 15  Bonus        66 non-null     str    
 16  Other        10 non-null     str    
 17  Class        83 non-null     str    
dtypes: float64(1), int64(2), str(15)
memory usage: 11.8 KB


In [80]:
dados_unlabled = pd.read_csv('AgeOfMythology_noClass.csv')

dados_unlabled.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          10 non-null     int64  
 1   Pop          10 non-null     int64  
 2   DmgHack      10 non-null     int64  
 3   DmgPierce    10 non-null     int64  
 4   DmgCrush     10 non-null     int64  
 5   Range        10 non-null     int64  
 6   HP           10 non-null     int64  
 7   ArmorHack    10 non-null     int64  
 8   ArmorPierce  10 non-null     int64  
 9   ArmorCrush   10 non-null     int64  
 10  Speed        10 non-null     float64
 11  Train time   10 non-null     int64  
 12  Favor        10 non-null     int64  
dtypes: float64(1), int64(12)
memory usage: 1.1 KB


2. Verifica quais colunas presentes nos dados rotulados estão ausentes nos não rotulados.

obs: Explicações do professor sobre as colunas:
- LOS: line of sights
- Bonus: contra quem ela é forte (negativo é fraco)
- Other: atributo especial
- Favor: outro recurso (custo de criação de unidade)

In [81]:
for column in dados_labled.columns:
    if column not in dados_unlabled.columns and column != 'Class':
        print(column)

Unit
Cost
LOS
Bonus
Other


Com a exceção de "Cost", vamos remover essas colunas pois não serão úteis para classificação. _"Cost"_ se refere ao custo de criação ed uma unidade, o qual pode fazer uso de diversos recursos diferentes. Um desses recursos é _"Favor"_ o qual está presente na tabela de dados não rotulados e será útil na classificação.

In [82]:
dados_labled['Cost'].head(7)

0      50F 2Fv
1    200W 15Fv
2    250F 22Fv
3    200F 16Fv
4    250F 28Fv
5    300W 28Fv
6    200G 25Fv
Name: Cost, dtype: str

3. Cria uma nova coluna Favor com os números seguidos por "Fv" da coluna Cost

In [83]:
dados_labled['Favor'] = (
    dados_labled['Cost'].str.extract(r'(\d+(?:\.\d+)?)\s*Fv', expand=False)
    .astype(float).fillna(0).round().astype('Int64')
)
dados_labled = dados_labled.drop(columns=['Cost'])

dados_labled[['Unit', 'Favor']].head(7)

,Unit,Favor
0,Pegasus,2
1,Centaur,15
2,Cyclops,22
3,Minotaur,16
4,Hydra,28
5,Manticore,28
6,Nemean Lion,25


4. Remove as coulnas desnecessárias para classificação

In [84]:
colunas_para_remover = dados_labled.columns.difference(dados_unlabled.columns).difference(["Class"])
dados_labled = dados_labled.drop(columns=colunas_para_remover)

dados_labled.columns

Index(['Age', 'Pop', 'DmgHack', 'DmgPierce', 'DmgCrush', 'Range', 'HP',
       'ArmorHack', 'ArmorPierce', 'ArmorCrush', 'Speed', 'Train time',
       'Class', 'Favor'],
      dtype='str')

Percebe-se algumas colunas do dataset rotulado estão com o tipo de dado incorreto (colunas str eram pra ser Int64). Conversão dos tipos de dados:

5. Conversão dos tipos de dados para que as das duas tableas fiquem iguais

In [85]:
# Algumas entradas têm caracteres extras junto ao valor numérico: "%" nas colunas de armadura, "*" nas de dano,
# "_" em HP e espaços invisíveis ( ) em Pop. Se fizer o casting direto, esses valores seriam perdidos (virariam NA).
# Vamos então remover tudo que não for dígito ou ponto decimal antes de converter. Valores como "--" e "none" continuam virando NA.
for coluna, serie in dados_unlabled.items():
    if coluna in dados_labled.columns and pd.api.types.is_integer_dtype(serie):
        coluna_limpa = dados_labled[coluna].astype(str).str.replace(r'[^\d.]', '', regex=True)
        coluna_convertida = pd.to_numeric(coluna_limpa, errors="coerce")
        dados_labled[coluna] = coluna_convertida.round().astype("Int64")

dados_labled.head()

,Age,Pop,DmgHack,DmgPierce,DmgCrush,Range,HP,ArmorHack,ArmorPierce,ArmorCrush,Speed,Train time,Class,Favor
0,1,2,<NA>,<NA>,<NA>,0,140,50,50,99,5.0,6,Myth Unit,2
1,2,3,<NA>,12,<NA>,12,220,25,35,80,5.0,19,Myth Unit,15
2,2,4,15,<NA>,12,2,600,40,50,80,3.2,20,Myth Unit,22
3,2,4,15,<NA>,10,2,300,60,50,80,4.0,20,Myth Unit,16
4,3,5,20,<NA>,10,3,800,60,40,80,4.0,20,Myth Unit,28


6. Tratamento de valores faltantes

Percebe-se que existem algumas colunas sem valores. Dependendo da coluna isso pode ser aceitável, por exemplo, uma unidade pode dar dano cortante (DmgHack) e dano esmagante (DmgCrush) mas não dar dano perfurante (DmgPierce). Nesse caso, o valor nulo pode ser zerado para ficar igual ao dataset não rotulado; ja em outros casos a falta de um valor tem que ser tratada para assegurar que a classificação seja feita devidamente.

obs: o dataset não rotulado não tem células nulas/sem valor

Vamos analisar o dataset não rotulado para ver como podemos tratar as células nulas no rotulado:

In [86]:
dados_unlabled.head(10)

,Age,Pop,DmgHack,DmgPierce,DmgCrush,Range,HP,ArmorHack,ArmorPierce,ArmorCrush,Speed,Train time,Favor
0,3,5,12,0,0,1,800,60,60,80,3.60,19,15
1,2,1,5,0,0,1,50,10,5,99,3.00,20,5
2,2,3,11,5,0,1,400,30,50,99,3.80,17,5
3,3,2,0,6,0,20,60,15,15,99,3.75,12,0
4,3,3,8,0,0,1,120,10,35,99,4.00,15,0
5,3,2,8,4,0,1,95,15,10,99,3.50,14,0
6,4,3,10,0,30,16,180,30,70,50,2.50,21,0
7,4,3,11,10,0,1,140,15,40,99,3.50,22,0
8,2,3,10,0,8,1,175,50,25,80,4.00,26,10
9,4,5,20,0,0,1,360,60,69,80,4.10,30,25


Analisando coluna por coluna:
- `Age`: As duas entradas com Age == NA no dataset rotulado são Heróis que não pertencem a uma era específica (Son of Osiris e Hero of Ragnarok). Como não existem entradas semelhantes no dataset não rotulado, serão **removidas**.
- `Pop` e `Train time`: Os valores faltantes restantes são todos de Heróis, que têm um perfil próprio de custo e tempo de treino. Em vez da mediana da coluna inteira, vamos preencher com a **mediana da mesma classe** (`Class`). Isso é seguro pois o dataset não rotulado não tem células nulas para preencher.
- `DmgHack`, `DmgPierce` e `DmgCrush`: `--` significa que a unidade não causa esse tipo de dano, então podemos setar valores faltantes como **Zero**. Valores com asterisco (ex: `11**`) são danos reais e já foram recuperados na conversão.
- `Range`: Só tem uma entrada do dataset rotulado sem range (`none`). Essa unidade também não dá dano nenhum. Como não existe nenhuma entrada no dataset de teste semelhante a essa, podemos considerá-la um **outlier** e **removê-la**.
- `HP`: o valor `"1100_"` (com underline no final) agora é recuperado automaticamente pela limpeza de caracteres feita na conversão de tipos.
- `ArmorHack`, `ArmorPierce` e `ArmorCrush`: Igual às colunas de dano.
- `Speed`: todas as células têm valores. Não é necessário fazer nada.


In [87]:
# Remove outliers: sem Age (Heróis sem era) ou sem Range (unidade sem dano)
dados_labled = dados_labled.dropna(subset=['Age', 'Range'])

# Dano e armadura ausentes significam "nenhum": zera
colunas_dano_armadura = ['DmgHack', 'DmgPierce', 'DmgCrush', 'ArmorHack', 'ArmorPierce', 'ArmorCrush']
dados_labled[colunas_dano_armadura] = dados_labled[colunas_dano_armadura].fillna(0)

# Pop e Train time: mediana da mesma classe
for coluna in ['Pop', 'Train time']:
    mediana_classe = dados_labled.groupby('Class')[coluna].transform('median')
    dados_labled[coluna] = dados_labled[coluna].fillna(mediana_classe).round().astype('Int64')

dados_labled = dados_labled.reset_index(drop=True)
dados_labled.isna().sum()

Age            0
Pop            0
DmgHack        0
DmgPierce      0
DmgCrush       0
Range          0
HP             0
ArmorHack      0
ArmorPierce    0
ArmorCrush     0
Speed          0
Train time     0
Class          0
Favor          0
dtype: int64

In [88]:
dados_labled.info()
dados_labled.head(80)

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          80 non-null     Int64  
 1   Pop          80 non-null     Int64  
 2   DmgHack      80 non-null     Int64  
 3   DmgPierce    80 non-null     Int64  
 4   DmgCrush     80 non-null     Int64  
 5   Range        80 non-null     Int64  
 6   HP           80 non-null     Int64  
 7   ArmorHack    80 non-null     Int64  
 8   ArmorPierce  80 non-null     Int64  
 9   ArmorCrush   80 non-null     Int64  
 10  Speed        80 non-null     float64
 11  Train time   80 non-null     Int64  
 12  Class        80 non-null     str    
 13  Favor        80 non-null     Int64  
dtypes: Int64(12), float64(1), str(1)
memory usage: 9.8 KB


,Age,Pop,DmgHack,DmgPierce,DmgCrush,Range,HP,ArmorHack,ArmorPierce,ArmorCrush,Speed,Train time,Class,Favor
0,1,2,0,0,0,0,140,50,50,99,5.0,6,Myth Unit,2
1,2,3,0,12,0,12,220,25,35,80,5.0,19,Myth Unit,15
2,2,4,15,0,12,2,600,40,50,80,3.2,20,Myth Unit,22
3,2,4,15,0,10,2,300,60,50,80,4.0,20,Myth Unit,16
4,3,5,20,0,10,3,800,60,40,80,4.0,20,Myth Unit,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2,2,0,6,0,12,290,30,20,10,6.0,14,Ship,0
76,3,2,20,0,0,1,240,20,20,75,7.0,12,Ship,0
77,3,3,0,0,6,18,480,10,50,10,4.8,17,Ship,0
78,3,5,15,0,5,1,900,50,70,80,6.0,10,Myth Unit,25
